In [ ]:
import pandas as pd

# Importer le fichier CSV
df = pd.read_csv("CAR DETAILS FROM CAR DEKHO.csv")

# Afficher les 5 premières lignes
df.head()
%pip install dagshub mlflow

In [ ]:
# Afficher le nombre de lignes et de colonnes
print("Dimensions :", df.shape)

# Afficher les informations sur les colonnes
df.info()

# Vérifier les valeurs manquantes
print("\nValeurs manquantes :")
print(df.isnull().sum())

# Compter les doublons
print("\nNombre de doublons :", df.duplicated().sum())

In [ ]:
import dagshub
import mlflow

dagshub.init(
    repo_owner="LYES475",
    repo_name="projet-ia-mlflow",
    mlflow=True
)

mlflow.set_experiment("Prix des voitures - Comparaison modèles")

In [ ]:
# Supprimer les doublons
df = df.drop_duplicates()

# Vérifier les dimensions après nettoyage
print("Dimensions après nettoyage :", df.shape)

# Vérifier qu'il ne reste aucun doublon
print("Doublons restants :", df.duplicated().sum())

In [ ]:
# Variable cible : prix de la voiture
y = df["selling_price"]

# Variables d'entrée : caractéristiques de la voiture
X = df.drop("selling_price", axis=1)

# Vérifier les dimensions
print("Variables d'entrée :", X.shape)
print("Variable cible :", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

# Séparer 60 % pour l'entraînement et 40 % pour le reste
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42
)

# Diviser les 40 % restants en validation et test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
)

# Afficher les dimensions
print("Entraînement :", X_train.shape)
print("Validation :", X_val.shape)
print("Test :", X_test.shape)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Identifier les colonnes contenant du texte
colonnes_cat = X.select_dtypes(include="object").columns

# Transformer le texte en nombres
preprocesseur = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore"), colonnes_cat)],
    remainder="passthrough"
)

# Apprendre la transformation sur l'entraînement
X_train_encoded = preprocesseur.fit_transform(X_train)

# Appliquer la même transformation à la validation et au test
X_val_encoded = preprocesseur.transform(X_val)
X_test_encoded = preprocesseur.transform(X_test)

print("Entraînement :", X_train_encoded.shape)
print("Validation :", X_val_encoded.shape)
print("Test :", X_test_encoded.shape)

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Normaliser les données pour KNN
scaler = StandardScaler(with_mean=False)

X_train_knn = scaler.fit_transform(X_train_encoded)
X_val_knn = scaler.transform(X_val_encoded)

# Créer et entraîner le modèle KNN
modele_knn = KNeighborsRegressor(n_neighbors=5)
modele_knn.fit(X_train_knn, y_train)

# Faire les prédictions sur la validation
y_pred_knn = modele_knn.predict(X_val_knn)

# Calculer les erreurs
mae_knn = mean_absolute_error(y_val, y_pred_knn)
rmse_knn = np.sqrt(mean_squared_error(y_val, y_pred_knn))

print("KNN - MAE :", round(mae_knn, 2))
print("KNN - RMSE :", round(rmse_knn, 2))

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Créer le modèle Random Forest
modele_rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

# Entraîner le modèle
modele_rf.fit(X_train_encoded, y_train)

# Faire les prédictions sur les données de validation
y_pred_rf = modele_rf.predict(X_val_encoded)

# Calculer les erreurs
mae_rf = mean_absolute_error(y_val, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_val, y_pred_rf))

# Afficher les résultats
print("Random Forest - MAE :", round(mae_rf, 2))
print("Random Forest - RMSE :", round(rmse_rf, 2))

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Créer le modèle Arbre de décision
modele_dt = DecisionTreeRegressor(random_state=42)

# Entraîner le modèle
modele_dt.fit(X_train_encoded, y_train)

# Faire les prédictions sur les données de validation
y_pred_dt = modele_dt.predict(X_val_encoded)

# Calculer les erreurs
mae_dt = mean_absolute_error(y_val, y_pred_dt)
rmse_dt = np.sqrt(mean_squared_error(y_val, y_pred_dt))

# Afficher les résultats
print("Arbre de décision - MAE :", round(mae_dt, 2))
print("Arbre de décision - RMSE :", round(rmse_dt, 2))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Créer le tableau de comparaison
resultats = pd.DataFrame({
    "Modèle": ["KNN", "Arbre de décision", "Random Forest"],
    "MAE": [mae_knn, mae_dt, mae_rf],
    "RMSE": [rmse_knn, rmse_dt, rmse_rf]
})

# Afficher les résultats
print(resultats.round(2))

# Afficher le meilleur modèle selon le MAE
meilleur = resultats.loc[resultats["MAE"].idxmin()]

print("\nMeilleur modèle :", meilleur["Modèle"])
print("MAE :", round(meilleur["MAE"], 2))

# Créer un graphique
plt.figure(figsize=(10, 5))

plt.bar(
    resultats["Modèle"],
    resultats["MAE"],
    color=["blue", "orange", "green"]
)

plt.title("Comparaison des modèles sur les données de validation")
plt.xlabel("Modèles")
plt.ylabel("MAE (INR)")

plt.show()

In [ ]:
import numpy as np

# Calculer l'erreur relative pour chaque voiture
erreur_relative = np.abs(
    (y_val.to_numpy() - y_pred_rf) / y_val.to_numpy()
)

# Métrique métier 1 : estimations avec une erreur de 20 % maximum
taux_estimations = np.mean(erreur_relative <= 0.20) * 100

# Métrique métier 2 : surestimations supérieures à 20 %
taux_surestimations = np.mean(
    y_pred_rf > y_val.to_numpy() * 1.20
) * 100

print("Taux d'estimations à ±20 % :", round(taux_estimations, 2), "%")
print("Taux de surestimations >20 % :", round(taux_surestimations, 2), "%")

In [ ]:
# Envoyer les résultats des trois modèles vers MLflow
essais = [
    ("KNN", modele_knn, mae_knn, rmse_knn),
    ("Random Forest", modele_rf, mae_rf, rmse_rf),
    ("Arbre de décision", modele_dt, mae_dt, rmse_dt),
]

for nom, modele, mae, rmse in essais:
    with mlflow.start_run(run_name=nom):
        mlflow.log_params(modele.get_params())
        mlflow.log_param("jeu_evaluation", "validation")
        mlflow.log_metric("MAE", float(mae))
        mlflow.log_metric("RMSE", float(rmse))

print("Les résultats des trois modèles ont été envoyés !")